In [217]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder

In [218]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jp797498e/twitter-entity-sentiment-analysis")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Lenovo\.cache\kagglehub\datasets\jp797498e\twitter-entity-sentiment-analysis\versions\2


In [219]:
df = pd.read_csv(r'C:\Users\Lenovo\OneDrive\Desktop\Data Structures and Algorithms\Deep Learning\Pytorch\2\twitter_training.csv')

In [220]:
df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [221]:
df.describe()

,2401
count,74681.000000
mean,6432.640149
std,3740.423819
min,1.000000
25%,3195.000000
50%,6422.000000
75%,9601.000000
max,13200.000000


In [222]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column                                                 Non-Null Count  Dtype 
---  ------                                                 --------------  ----- 
 0   2401                                                   74681 non-null  int64 
 1   Borderlands                                            74681 non-null  object
 2   Positive                                               74681 non-null  object
 3   im getting on borderlands and i will murder you all ,  73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [223]:
df['Positive'].value_counts()

Positive
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64

In [224]:
df.drop(columns=['2401','Borderlands'],inplace=True)

In [225]:
df.head()

,Positive,"im getting on borderlands and i will murder you all ,"
0,Positive,I am coming to the borders and I will kill you...
1,Positive,im getting on borderlands and i will kill you ...
2,Positive,im coming on borderlands and i will murder you...
3,Positive,im getting on borderlands 2 and i will murder ...
4,Positive,im getting into borderlands and i can murder y...


In [226]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")


In [227]:
print(df.columns)
# ['text', 'label']


Index(['Positive', 'im getting on borderlands and i will murder you all ,'], dtype='object')


In [228]:
df = df.rename(columns = {
    'im getting on borderlands and i will murder you all ,': 'text',
    'Positive': 'sentiment'
})

In [229]:
df

,sentiment,text
0,Positive,I am coming to the borders and I will kill you...
1,Positive,im getting on borderlands and i will kill you ...
2,Positive,im coming on borderlands and i will murder you...
3,Positive,im getting on borderlands 2 and i will murder ...
4,Positive,im getting into borderlands and i can murder y...
...,...,...
74676,Positive,Just realized that the Windows partition of my...
74677,Positive,Just realized that my Mac window partition is ...
74678,Positive,Just realized the windows partition of my Mac ...
74679,Positive,Just realized between the windows partition of...


In [230]:
Encoder = LabelEncoder()

In [231]:
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
texts = df['text'].fillna("").astype(str).tolist()

encodings = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

df['label_encoded'] = Encoder.fit_transform(df['sentiment'])

In [232]:
df

,sentiment,text,label_encoded
0,Positive,I am coming to the borders and I will kill you...,3
1,Positive,im getting on borderlands and i will kill you ...,3
2,Positive,im coming on borderlands and i will murder you...,3
3,Positive,im getting on borderlands 2 and i will murder ...,3
4,Positive,im getting into borderlands and i can murder y...,3
...,...,...,...
74676,Positive,Just realized that the Windows partition of my...,3
74677,Positive,Just realized that my Mac window partition is ...,3
74678,Positive,Just realized the windows partition of my Mac ...,3
74679,Positive,Just realized between the windows partition of...,3


In [233]:
encodings

{'input_ids': tensor([[  101,  1045,  2572,  ...,     0,     0,     0],
        [  101, 10047,  2893,  ...,     0,     0,     0],
        [  101, 10047,  2746,  ...,     0,     0,     0],
        ...,
        [  101,  2074,  3651,  ...,     0,     0,     0],
        [  101,  2074,  3651,  ...,     0,     0,     0],
        [  101,  2074,  2066,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

In [234]:
encodings[0]

Encoding(num_tokens=128, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [235]:
inputs_ids = encodings['input_ids']
attention_mask = encodings['attention_mask']

In [236]:
from sklearn.model_selection import train_test_split


In [237]:
X = inputs_ids
y = df['label_encoded']

In [238]:
X

tensor([[  101,  1045,  2572,  ...,     0,     0,     0],
        [  101, 10047,  2893,  ...,     0,     0,     0],
        [  101, 10047,  2746,  ...,     0,     0,     0],
        ...,
        [  101,  2074,  3651,  ...,     0,     0,     0],
        [  101,  2074,  3651,  ...,     0,     0,     0],
        [  101,  2074,  2066,  ...,     0,     0,     0]])

In [239]:
X.shape

torch.Size([74681, 128])

In [240]:
y

0        3
1        3
2        3
3        3
4        3
        ..
74676    3
74677    3
74678    3
74679    3
74680    3
Name: label_encoded, Length: 74681, dtype: int64

In [241]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [242]:
X_train

tensor([[  101,  2026,  8673,  ...,     0,     0,     0],
        [  101,  1045,  2442,  ...,     0,     0,     0],
        [  101,  2449, 12476,  ...,     0,     0,     0],
        ...,
        [  101,  2054, 10424,  ...,     0,     0,     0],
        [  101,  3675,  8653,  ...,     0,     0,     0],
        [  101,  8398,  2074,  ...,     0,     0,     0]])

In [243]:
y_train

8581     2
71533    3
67251    2
41061    3
16591    3
        ..
37194    2
6265     2
54886    1
860      1
15795    2
Name: label_encoded, Length: 59744, dtype: int64

In [244]:
X_test

tensor([[  101,  2253,  2000,  ...,     0,     0,     0],
        [  101, 10930,  2023,  ...,     0,     0,     0],
        [  101,  3477,  3086,  ...,     0,     0,     0],
        ...,
        [  101,  4593, 18368,  ...,     0,     0,     0],
        [  101,  4593,  1996,  ...,     0,     0,     0],
        [  101,  4012, 12031,  ...,     0,     0,     0]])

In [245]:
y_test

34877    0
21704    3
47008    1
7969     0
454      3
        ..
52360    0
57296    3
35884    3
59060    1
4740     2
Name: label_encoded, Length: 14937, dtype: int64

In [246]:
from torch.utils.data import DataLoader, Dataset

In [247]:
class Twitte(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = torch.tensor(y.values, dtype = torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        
        return {
            'input_ids' : self.X[index],
            'labels': self.y[index]
        }


In [248]:
train_dataset = Twitte(X_train,y_train)
test_dataset = Twitte(X_test,y_test)

In [249]:
train_loader = DataLoader(train_dataset,batch_size = 8,shuffle = True)
test_loader = DataLoader(test_dataset,batch_size=8,shuffle= True)

In [253]:
class SimpleLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim*2, output_dim)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        return self.fc(last_hidden)
 


In [254]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [255]:
vocab_size = tokenizer.vocab_size
embed_dim = 32
hidden_dim = 64



output_dim = len(Encoder.classes_)

model = SimpleLSTM(vocab_size, 32, 64, output_dim)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model.to(device)

SimpleLSTM(
  (embedding): Embedding(30522, 32)
  (lstm): LSTM(32, 64, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=128, out_features=4, bias=True)
)

In [256]:
epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")


Epoch 1 | Loss: 1.3674
Epoch 2 | Loss: 1.3668
Epoch 3 | Loss: 1.3666
Epoch 4 | Loss: 1.3665
Epoch 5 | Loss: 1.3205


In [262]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids)
        preds = torch.argmax(outputs, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("Train Accuracy:", correct / total)


TypeError: argmax(): argument 'input' (position 1) must be Tensor, not SequenceClassifierOutput

In [258]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids)
        preds = torch.argmax(outputs, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("Test Accuracy:", correct / total)


Test Accuracy: 0.4230434491531097
